# CircuitCost: final experiment

Predict GEMM implementation LUT count from 11 HLS features for
`xczu9eg-ffvb1156-2-i`. Install the project with `make setup` and `make notebooks`,
select its Python environment, and restore the CSVs as described in the README.

This notebook uses relative paths, a checked one-to-one design join, fixed split
and initialization seeds, training-only feature scaling, and best-validation
checkpoint restoration. Targets stay in LUT units. The published test set has
already been evaluated; the test cell below is opt-in for reproducing that
evaluation, not for selecting new configurations.

In [1]:
import hashlib
import json
import platform
from importlib.metadata import version

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from circuitcost.data import (
    DATASET_REVISION, DATASET_SUBDIR, FEATURE_COLUMNS, TARGET_COLUMN,
    find_project_root, load_designs,
)
from circuitcost.evaluation import errors_by_lut_range, regression_metrics
from circuitcost.models import NeuralNetworks
from circuitcost.train import train_model

In [2]:
project_root = find_project_root()
data_dir = project_root / DATASET_SUBDIR
artifact_dir = project_root / "artifacts" / "mlp_32_16_seed42"

SPLIT_SEED = 42
MODEL_SEED = 42
MAX_EPOCHS = 50000
LEARNING_RATE = 0.000001
feature_columns = list(FEATURE_COLUMNS)

# A fresh checkout trains once; existing local checkpoints are loaded by default.
RUN_TRAINING = not (artifact_dir / "weights.npz").exists()
RUN_SEED_CHECKS = False
EVALUATE_TEST = False

## Match designs before splitting

`load_designs` explicitly joins on `prj`, rejects duplicate/missing/unmatched IDs,
and checks finite features and targets. It preserves HLS file order and uses
`prj` as the index, so split membership can be recorded by design rather than row
position. It does not include implementation measurements among the inputs.

In [3]:
designs = load_designs(data_dir)
X = designs[feature_columns].copy()
y = designs[TARGET_COLUMN].copy()
display(designs.head())
print("Matched designs:", len(designs), "Features:", X.shape[1])

csv_hashes = {
    name: hashlib.sha256((data_dir / name).read_bytes()).hexdigest()
    for name in ["post_hls_info.csv", "post_implementation_info.csv"]
}

,prj,clk_target(ns),clk_estimated(ns),clk_uncertainty(ns),bram,dsp,ff,lut,c_num_arith,c_num_logic,rtl_num_arith,rtl_num_logic,Total LUTs
prj,,,,,,,,,,,,,
io1_l2n1n1_l4n1n1,io1_l2n1n1_l4n1n1,10.0,8.419,1.25,32,5,758,1239,22,9,19,9,645
io1_l2n1n1_l4n1p1,io1_l2n1n1_l4n1p1,10.0,8.419,1.25,32,5,933,1356,23,9,20,9,656
io1_l2n1n1_l4p1n1,io1_l2n1n1_l4p1n1,10.0,9.298,1.25,32,5,4578,3914,146,71,17,71,4131
io1_l2n1p1_l4n1n1,io1_l2n1p1_l4n1n1,10.0,8.419,1.25,32,5,845,1416,24,12,21,12,813
io1_l2n1p1_l4n1p1,io1_l2n1p1_l4n1p1,10.0,8.419,1.25,32,5,1020,1519,25,12,22,12,833


Matched designs: 531 Features: 11


In [4]:
X_train, X_remaining, y_train, y_remaining = train_test_split(
    X, y, test_size=0.30, random_state=SPLIT_SEED
)
X_val, X_test, y_val, y_test = train_test_split(
    X_remaining, y_remaining, test_size=0.50, random_state=SPLIT_SEED
)
split_ids = {
    name: frame.index.tolist()
    for name, frame in [("train", X_train), ("validation", X_val), ("test", X_test)]
}
assert set(X_train.index).isdisjoint(X_val.index)
assert set(X_train.index).isdisjoint(X_test.index)
assert set(X_val.index).isdisjoint(X_test.index)
print({name: len(ids) for name, ids in split_ids.items()})

{'train': 371, 'validation': 80, 'test': 80}


## Train or load the frozen candidate

The model is `11 -> 32 -> ReLU -> 16 -> ReLU -> 1`. The implementation preserves
the recorded random-normal weight and bias initialization. One epoch makes one
full-batch update. Validation selects copied parameters; training runs to its
budget and restores the selected checkpoint afterward. This is not early stopping.

Set `RUN_TRAINING=True` to repeat the seed-42 experiment. Loading an existing
checkpoint checks feature order, split IDs, and the fitted scaler against the
current data. It does not refit preprocessing on validation or test examples.

In [5]:
reference_scaler = StandardScaler().fit(X_train)

if RUN_TRAINING:
    scaler = reference_scaler
else:
    metadata = json.loads((artifact_dir / "metadata.json").read_text())
    assert metadata["features"] == feature_columns
    assert metadata["hidden_layers"] == [32, 16]
    assert metadata["split_design_ids"] == split_ids
    if "csv_sha256" in metadata:
        assert metadata["csv_sha256"] == csv_hashes
    scaler = joblib.load(artifact_dir / "scaler.joblib")
    np.testing.assert_allclose(scaler.mean_, reference_scaler.mean_)
    np.testing.assert_allclose(scaler.scale_, reference_scaler.scale_)

X_train_scaled = scaler.transform(X_train)
X_val_scaled = scaler.transform(X_val)

np.random.seed(MODEL_SEED)
model = NeuralNetworks(n_inputs=X_train.shape[1])

if RUN_TRAINING:
    history = train_model(
        model, X_train_scaled, y_train, X_val_scaled, y_val,
        epochs=MAX_EPOCHS, learning_rate=LEARNING_RATE,
    )
    linear_model = LinearRegression().fit(X_train_scaled, y_train)
    best_epoch = history["best_epoch"]
else:
    with np.load(artifact_dir / "weights.npz", allow_pickle=False) as checkpoint:
        dense_layers = [layer for layer in model.layers if hasattr(layer, "weights")]
        for i, layer in enumerate(dense_layers):
            weights = checkpoint[f"weights_{i}"]
            biases = checkpoint[f"biases_{i}"]
            assert weights.shape == layer.weights.shape
            assert biases.shape == layer.biases.shape
            layer.weights = weights.copy()
            layer.biases = biases.copy()
    linear_model = joblib.load(artifact_dir / "linear_model.joblib")
    history = None
    best_epoch = metadata["best_epoch"]

validation_predictions = {
    "HLS estimate": X_val["lut"].to_numpy(),
    "Linear regression": linear_model.predict(X_val_scaled),
    "Custom MLP": model.forward(X_val_scaled).ravel(),
}
validation_results = pd.DataFrame({
    name: regression_metrics(y_val, predicted)
    for name, predicted in validation_predictions.items()
}).T
display(validation_results)
print("Selected epoch:", best_epoch)

if history is not None:
    expected_loss = history["best_val_loss"]
else:
    expected_loss = metadata["validation_metrics"]["RMSE"] ** 2
assert np.isclose(validation_results.loc["Custom MLP", "RMSE"] ** 2, expected_loss)

,MAE,RMSE,R2
HLS estimate,5758.975000,9836.975108,0.384620
Linear regression,936.894108,1595.877056,0.983804
Custom MLP,548.199200,891.686089,0.994944


Selected epoch: 2336


In [6]:
if history is not None:
    plt.figure(figsize=(10, 5))
    plt.plot(np.sqrt(history["train_loss"]), label="Training", alpha=0.5, linewidth=0.7)
    plt.plot(np.sqrt(history["val_loss"]), label="Validation", alpha=0.5, linewidth=0.7)
    plt.xlabel("Epoch")
    plt.ylabel("RMSE (LUTs)")
    plt.legend()
    plt.show()
else:
    print("Checkpoint loaded; historical learning curves were not saved with it.")

Checkpoint loaded; historical learning curves were not saved with it.


## Save a newly trained candidate

The weights, fitted input scaler, linear baseline, feature order, split IDs,
dataset checksums, package versions, and selected epoch are local artifacts.
The artifacts directory is ignored by Git. A loaded checkpoint is not overwritten.

In [7]:
if RUN_TRAINING:
    artifact_dir.mkdir(parents=True, exist_ok=True)
    parameters = {}
    for i, layer in enumerate(layer for layer in model.layers if hasattr(layer, "weights")):
        parameters[f"weights_{i}"] = layer.weights.copy()
        parameters[f"biases_{i}"] = layer.biases.copy()
    np.savez_compressed(artifact_dir / "weights.npz", **parameters)
    joblib.dump(scaler, artifact_dir / "scaler.joblib")
    joblib.dump(linear_model, artifact_dir / "linear_model.joblib")
    metadata = {
        "features": feature_columns,
        "hidden_layers": [32, 16],
        "target": TARGET_COLUMN,
        "target_scaling": "none",
        "seed": MODEL_SEED,
        "learning_rate": LEARNING_RATE,
        "max_epochs": MAX_EPOCHS,
        "best_epoch": best_epoch,
        "split_random_state": SPLIT_SEED,
        "split_design_ids": split_ids,
        "dataset_revision": DATASET_REVISION,
        "csv_sha256": csv_hashes,
        "python_version": platform.python_version(),
        "packages": {name: version(name) for name in [
            "numpy", "pandas", "scikit-learn", "scipy", "matplotlib", "joblib"
        ]},
        "validation_metrics": validation_results.loc["Custom MLP"].to_dict(),
    }
    (artifact_dir / "metadata.json").write_text(json.dumps(metadata, indent=2))
    print("Saved candidate:", artifact_dir.relative_to(project_root))

## Optional initialization-seed check

Only initialization changes; data splits, scaler, architecture, and learning rate
stay fixed. This runs two additional 50,000-epoch experiments. Seed 42 remains
the predefined candidate; test scores do not select a seed. Recorded seed
results are available in `reports/recorded_seed_results.csv`.

In [8]:
if RUN_SEED_CHECKS:
    records = [{"seed": MODEL_SEED, "status": "ok", "best_epoch": best_epoch,
                **validation_results.loc["Custom MLP"].to_dict()}]
    for seed in [7, 21]:
        np.random.seed(seed)
        repeat_model = NeuralNetworks(X_train.shape[1])
        try:
            repeat_history = train_model(
                repeat_model, X_train_scaled, y_train, X_val_scaled, y_val,
                epochs=MAX_EPOCHS, learning_rate=LEARNING_RATE,
            )
            predicted = repeat_model.forward(X_val_scaled).ravel()
            records.append({"seed": seed, "status": "ok",
                            "best_epoch": repeat_history["best_epoch"],
                            **regression_metrics(y_val, predicted)})
        except ValueError as error:
            records.append({"seed": seed, "status": "failed", "error": str(error)})
    seed_results = pd.DataFrame(records)
    display(seed_results)
    artifact_dir.mkdir(parents=True, exist_ok=True)
    seed_results.to_csv(artifact_dir / "seed_results.csv", index=False)
else:
    print("Additional seed experiments skipped.")

Additional seed experiments skipped.


## Reproduce the recorded test evaluation

Set `EVALUATE_TEST=True` only when reproducing the frozen evaluation. Do not use
these results to choose a new architecture or seed. Actual-LUT ranges below are
descriptive groups (`<5,000`, `5,000–14,999`, `>=15,000`), not model-selection
rules: actual implementation usage is unknown at prediction time. See the final
findings for the recorded trade-off and the limits of this single-benchmark split.

In [9]:
if EVALUATE_TEST:
    X_test_scaled = scaler.transform(X_test)
    test_predictions = {
        "HLS estimate": X_test["lut"].to_numpy(),
        "Linear regression": linear_model.predict(X_test_scaled),
        "Custom MLP": model.forward(X_test_scaled).ravel(),
    }
    test_results = pd.DataFrame({
        name: regression_metrics(y_test, predicted)
        for name, predicted in test_predictions.items()
    }).T
    range_results = errors_by_lut_range(y_test, test_predictions)
    display(test_results)
    display(range_results)
    artifact_dir.mkdir(parents=True, exist_ok=True)
    test_results.to_csv(artifact_dir / "test_metrics.csv")
    range_results.to_csv(artifact_dir / "test_errors_by_lut_range.csv", index=False)
else:
    print("Test evaluation skipped. Published results are in docs/FINAL_FINDINGS.md.")

Test evaluation skipped. Published results are in docs/FINAL_FINDINGS.md.
